# BirdCLEF 2026 — Pantanal Wildlife Audio Classification
### Inference Notebook

**How this works:**
1. Loads trained model weights from an uploaded Kaggle dataset
2. Slides a 5-second window over every test soundscape
3. Predicts probability of each of 234 species per window
4. Writes `submission.csv` to `/kaggle/working/`

**Expected Kaggle dataset structure:**
```
/kaggle/input/birdclef-2026/          ← competition data (auto-mounted)
/kaggle/input/birdclf2026-weights/    ← your uploaded checkpoints
    best_fold0.pt
    best_fold1.pt   (if available)
    ...
```

In [1]:
# Install any packages not pre-installed on Kaggle
# librosa and timm are usually available — uncomment if needed
# !pip install -q librosa timm

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import librosa  # kept only for resample fallback
import timm
from tqdm.notebook import tqdm

print(f'PyTorch    : {torch.__version__}')
print(f'Torchaudio : {torchaudio.__version__}')
print(f'Device     : {"cuda" if torch.cuda.is_available() else "cpu"}')

PyTorch    : 2.10.0+cpu
Torchaudio : 2.10.0+cpu
Device     : cpu


## Configuration
All paths and hyperparameters in one place — edit here if needed.

In [3]:
import os

# ── Show full folder tree so we always know exact paths ─────────────
INPUT_DIR  = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working')

print('Full /kaggle/input structure:')
for root, dirs, files in os.walk(INPUT_DIR):
    depth = str(root).count(os.sep) - str(INPUT_DIR).count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    if depth >= 2:   # stop drilling into audio files
        dirs.clear()

# ── Find competition folder — search all subdirs for taxonomy.csv ───
taxonomy_hits = list(INPUT_DIR.rglob('taxonomy.csv'))
assert len(taxonomy_hits) > 0, (
    'taxonomy.csv not found anywhere under /kaggle/input. '
    'Add BirdCLEF+ 2026 competition data via + Add Input.'
)
COMP_DIR = taxonomy_hits[0].parent
print(f'\nCompetition data : {COMP_DIR}')

# ── Find weights folder — search for any best_fold*.pt ──────────────
weight_hits = list(INPUT_DIR.rglob('best_fold*.pt'))
assert len(weight_hits) > 0, (
    'No best_fold*.pt found under /kaggle/input. '
    'Add birdclf2026-weights dataset via + Add Input.'
)
WEIGHTS_DIR = weight_hits[0].parent
print(f'Weights          : {WEIGHTS_DIR}')

TAXONOMY_CSV    = COMP_DIR / 'taxonomy.csv'
TEST_AUDIO_DIR  = COMP_DIR / 'test_soundscapes'
TRAIN_AUDIO_DIR = COMP_DIR / 'train_soundscapes'
SAMPLE_SUB_CSV  = COMP_DIR / 'sample_submission.csv'

# ── Audio constants ─────────────────────────────────────────────────
SAMPLE_RATE = 32_000
WINDOW_SECS = 5
WINDOW_LEN  = SAMPLE_RATE * WINDOW_SECS
N_FFT       = 1024
HOP_LENGTH  = 320
N_MELS      = 128
FMIN        = 50
FMAX        = 14_000

# ── Inference settings ──────────────────────────────────────────────
BATCH_SIZE  = 64
NUM_WORKERS = 0      # 0 = main process only — avoids multiprocessing hangs in Kaggle
USE_TTA     = False  # disabled to reduce inference time
N_TTA       = 0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {DEVICE}')

Full /kaggle/input structure:
input/
  competitions/
    birdclef-2026/
  datasets/
    ramkumarsundaram24/

Competition data : /kaggle/input/competitions/birdclef-2026
Weights          : /kaggle/input/datasets/ramkumarsundaram24/birdclf2026-weights
Device           : cpu


## Species List
Load from taxonomy.csv — this defines the column order in submission.csv.

In [4]:
taxonomy_df  = pd.read_csv(TAXONOMY_CSV)
SPECIES_LIST = taxonomy_df['primary_label'].astype(str).tolist()
NUM_CLASSES  = len(SPECIES_LIST)
print(f'Species: {NUM_CLASSES}')
print(f'First 5: {SPECIES_LIST[:5]}')

Species: 234
First 5: ['1161364', '116570', '1176823', '1491113', '1595929']


## Audio & Spectrogram Utilities

In [5]:
# ── GPU-accelerated mel spectrogram transform ───────────────────────
# Built once and reused for every window — much faster than librosa per-call
MEL_TRANSFORM = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=FMIN,
    f_max=FMAX,
).to(DEVICE)

DB_TRANSFORM = T.AmplitudeToDB(top_db=80).to(DEVICE)


def audio_to_melspec(audio_tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert (samples,) float32 tensor → normalised log-mel spectrogram tensor.
    Runs on GPU when DEVICE='cuda'.
    """
    mel = MEL_TRANSFORM(audio_tensor.to(DEVICE))   # (n_mels, time)
    db  = DB_TRANSFORM(mel)                         # log scale
    # Normalise to [0, 1]
    db_min, db_max = db.min(), db.max()
    db = (db - db_min) / (db_max - db_min + 1e-6)
    return db  # (n_mels, time)


def load_audio(filepath: Path, max_duration: int = 900) -> torch.Tensor:
    """
    Load audio with torchaudio (faster than librosa for ogg).
    Returns mono float32 tensor at SAMPLE_RATE, capped at max_duration seconds.
    """
    waveform, sr = torchaudio.load(str(filepath))   # (channels, samples)
    waveform = waveform.mean(dim=0)                  # mono

    # Resample if needed
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)

    # Cap length
    max_samples = max_duration * SAMPLE_RATE
    if len(waveform) > max_samples:
        waveform = waveform[:max_samples]

    return waveform  # (samples,)


def spec_augment(spec: torch.Tensor,
                 num_freq: int = 2, freq_size: int = 15,
                 num_time: int = 2, time_size: int = 30) -> torch.Tensor:
    """SpecAugment on a (n_mels, time) tensor."""
    spec = spec.clone()
    n_mels, n_time = spec.shape
    for _ in range(num_freq):
        f0 = random.randint(0, n_mels - freq_size)
        spec[f0:f0 + freq_size, :] = 0.0
    for _ in range(num_time):
        t0 = random.randint(0, max(0, n_time - time_size))
        spec[:, t0:t0 + time_size] = 0.0
    return spec


print('Audio utilities ready (torchaudio backend).')

Audio utilities ready (torchaudio backend).


## Soundscape Dataset
Slices each test recording into non-overlapping 5-second windows.
Row ID format: `{filename_stem}_{end_second}`

In [6]:
class SoundscapeDataset(Dataset):
    """
    Loads one soundscape file and returns all 5-second windows.
    Uses torchaudio for fast ogg loading.
    row_id format: {filename_stem}_{end_second}
    """
    def __init__(self, filepath: Path):
        self.stem = filepath.stem
        self.windows = []

        try:
            audio = load_audio(filepath)  # torchaudio-based, fast
        except Exception as e:
            print(f'  ⚠️  Could not load {filepath.name}: {e} — skipping')
            return

        n_samples = len(audio)
        for start in range(0, n_samples, WINDOW_LEN):
            chunk = audio[start:start + WINDOW_LEN]
            if len(chunk) < WINDOW_LEN:
                chunk = torch.nn.functional.pad(chunk, (0, WINDOW_LEN - len(chunk)))
            end_sec = (start // WINDOW_LEN + 1) * WINDOW_SECS
            self.windows.append((chunk, end_sec))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        audio_chunk, end_sec = self.windows[idx]
        spec = audio_to_melspec(audio_chunk).cpu()   # (n_mels, time)
        return {
            'spectrogram': spec.unsqueeze(0),        # (1, n_mels, time)
            'row_id': f'{self.stem}_{end_sec}',
        }

## Model Definition

In [7]:
class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes: int, model_name: str = 'efficientnet_b3',
                 pretrained: bool = False, drop_rate: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, in_chans=1,
            num_classes=0, global_pool='avg', drop_rate=drop_rate,
        )
        feature_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 512), nn.ReLU(),
            nn.Dropout(drop_rate), nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


class CNN14Block(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
        )
        self.pool = nn.AvgPool2d(2)

    def forward(self, x):
        return self.pool(self.conv(x))


class CNN14Classifier(nn.Module):
    def __init__(self, num_classes: int, drop_rate: float = 0.3):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(1)
        self.layers = nn.Sequential(
            CNN14Block(1, 64),    CNN14Block(64, 128),
            CNN14Block(128, 256), CNN14Block(256, 512),
            CNN14Block(512, 1024), CNN14Block(1024, 2048),
        )
        self.dropout = nn.Dropout(drop_rate)
        self.head = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.bn0(x)
        x = self.layers(x).mean(dim=[2, 3])
        return self.head(self.dropout(x))


def detect_model_type(state_dict: dict) -> str:
    """Auto-detect model type from checkpoint keys."""
    keys = list(state_dict.keys())
    if any('backbone.conv_stem' in k or 'backbone.blocks' in k for k in keys):
        return 'efficientnet_b3'
    elif any('bn0' in k or 'layers.0' in k for k in keys):
        return 'cnn14'
    else:
        raise ValueError(f'Cannot detect model type from keys: {keys[:5]}')


def load_model(checkpoint_path: Path, num_classes: int, device):
    """Load a trained checkpoint — auto-detects EfficientNet vs CNN14."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint['model']

    model_type = detect_model_type(state_dict)
    print(f'  {checkpoint_path.name} → detected as {model_type}')

    if model_type == 'efficientnet_b3':
        model = EfficientNetClassifier(num_classes, pretrained=False)
    else:
        model = CNN14Classifier(num_classes)

    model.load_state_dict(state_dict)
    model.eval()
    return model.to(device)


print('Model classes defined.')

Model classes defined.


## Load Checkpoints
Loads all available fold checkpoints from your uploaded dataset and ensembles them.

In [8]:
checkpoints = sorted(WEIGHTS_DIR.glob('best_fold*.pt'))
print(f'Found {len(checkpoints)} checkpoint(s):')
for c in checkpoints:
    print(f'  {c.name}')

assert len(checkpoints) > 0, (
    f'No checkpoints found in {WEIGHTS_DIR}. '
    'Upload your best_fold*.pt files as a Kaggle dataset named birdclf2026-weights.'
)

models = []
for ckpt in checkpoints:
    m = load_model(ckpt, NUM_CLASSES, DEVICE)
    models.append(m)

print(f'\nEnsemble ready: {len(models)} model(s)')

Found 2 checkpoint(s):
  best_fold0.pt
  best_fold0_cnn14.pt
  best_fold0.pt → detected as efficientnet_b3
  best_fold0_cnn14.pt → detected as cnn14

Ensemble ready: 2 model(s)


## Inference
Slides a 5-second window over every test soundscape and averages predictions across all models + TTA.

In [9]:
@torch.no_grad()
def predict_batch(models, specs, device, use_tta=False, n_tta=0):
    """Run all models on a batch, average their sigmoid outputs."""
    specs = specs.to(device)
    all_probs = []

    for model in models:
        probs = torch.sigmoid(model(specs)).cpu().numpy()
        all_probs.append(probs)

        if use_tta:
            for _ in range(n_tta):
                aug = torch.stack([
                    torch.from_numpy(
                        spec_augment(s.squeeze(0).numpy())
                    ).unsqueeze(0)
                    for s in specs.cpu()
                ]).to(device)
                all_probs.append(torch.sigmoid(model(aug)).cpu().numpy())

    return np.mean(all_probs, axis=0)


def run_inference(models, test_dir, species_list, device,
                  batch_size=64, use_tta=False, n_tta=0, max_files=None):
    """Process all test soundscapes, return full predictions DataFrame."""
    audio_exts = {'.ogg', '.flac', '.wav', '.mp3'}
    files = sorted([f for f in test_dir.rglob('*') if f.suffix in audio_exts])

    if max_files is not None:
        files = files[:max_files]
        print(f'[DEV MODE] Limiting to {max_files} files for pipeline testing.')

    print(f'Test soundscapes found: {len(files)}')

    all_row_ids, all_probs = [], []

    for filepath in tqdm(files, desc='Soundscapes'):
        try:
            ds = SoundscapeDataset(filepath)
            if len(ds) == 0:
                print(f'  Skipping empty: {filepath.name}')
                continue

            loader = DataLoader(ds, batch_size=batch_size,
                                shuffle=False, num_workers=0)

            for batch in loader:
                probs = predict_batch(models, batch['spectrogram'],
                                      device, use_tta, n_tta)
                all_row_ids.extend(batch['row_id'])
                all_probs.append(probs)

        except Exception as e:
            print(f'  ⚠️  Error on {filepath.name}: {e} — skipping')
            continue

    probs_matrix = np.concatenate(all_probs)
    df = pd.DataFrame(probs_matrix, columns=species_list)
    df.insert(0, 'row_id', all_row_ids)
    return df


print('Inference functions ready.')

Inference functions ready.


In [10]:
# ── Check if test soundscapes exist (only populated during official submission) ──
audio_exts = {'.ogg', '.flac', '.wav', '.mp3'}
test_files = [f for f in TEST_AUDIO_DIR.rglob('*') if f.suffix in audio_exts]

if len(test_files) == 0:
    print('⚠️  test_soundscapes/ is empty.')
    print('   This is normal during interactive editing — Kaggle only fills it at submission time.')
    print('   Falling back to train_soundscapes/ to verify the pipeline works...\n')
    INFER_DIR = COMP_DIR / 'train_soundscapes'
    IS_TEST_RUN = False
else:
    print(f'✓ Found {len(test_files)} test soundscape(s) — running real inference.')
    INFER_DIR = TEST_AUDIO_DIR
    IS_TEST_RUN = True

# ── DEV MODE: limit files for quick pipeline testing ───────────────
# Set to None before Save Version → Commit for real submission
MAX_FILES = 10 if not IS_TEST_RUN else None

# Run inference
predictions_df = run_inference(
    models,
    test_dir=INFER_DIR,
    species_list=SPECIES_LIST,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    use_tta=USE_TTA,
    n_tta=N_TTA,
    max_files=MAX_FILES,
)

print(f'\nPredictions shape: {predictions_df.shape}')
predictions_df.head(3)

⚠️  test_soundscapes/ is empty.
   This is normal during interactive editing — Kaggle only fills it at submission time.
   Falling back to train_soundscapes/ to verify the pipeline works...

[DEV MODE] Limiting to 10 files for pipeline testing.
Test soundscapes found: 10


Soundscapes:   0%|          | 0/10 [00:00<?, ?it/s]


Predictions shape: (130, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.007431,0.289902,0.000315,0.069430,0.008472,5.221167e-07,0.005302,0.000841,0.002990,...,0.000432,0.012881,0.003394,0.000499,0.000055,0.006636,0.000044,0.001778,0.000031,0.004524
1,BC2026_Train_0001_S08_20250606_030007_10,0.006724,0.280341,0.000249,0.058997,0.009700,3.849934e-07,0.004129,0.000708,0.002880,...,0.000334,0.011540,0.002853,0.000423,0.000031,0.005639,0.000030,0.001611,0.000025,0.004076
2,BC2026_Train_0001_S08_20250606_030007_15,0.006187,0.285313,0.000362,0.071405,0.010047,5.648650e-07,0.004731,0.000813,0.004203,...,0.000590,0.013397,0.004248,0.000614,0.000051,0.008098,0.000046,0.002352,0.000040,0.005106


## Write Submission
Align columns exactly to sample_submission.csv and save.

In [11]:
# Load sample submission — this defines the exact rows AND columns Kaggle expects
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
expected_columns = sample_sub.columns.tolist()   # ['row_id', species1, ...]
prob_cols = [c for c in expected_columns if c != 'row_id']

print(f'Sample submission : {len(sample_sub)} rows × {len(expected_columns)} columns')
print(f'Our predictions   : {len(predictions_df)} rows')
print(f'Sample row_id sample: {sample_sub["row_id"].iloc[0]}')
print(f'Our row_id sample  : {predictions_df["row_id"].iloc[0]}')

# Start from sample_submission (guaranteed correct format) and fill in our predictions
submission = sample_sub.copy()
pred_indexed = predictions_df.set_index('row_id')

matched = 0
for col in prob_cols:
    if col in pred_indexed.columns:
        vals = submission['row_id'].map(pred_indexed[col])
        matched_count = vals.notna().sum()
        submission[col] = vals.fillna(0.0).astype(np.float32)
        matched = matched_count  # same for all cols
    else:
        submission[col] = 0.0

print(f'Matched rows      : {matched} / {len(sample_sub)}')

# Save
output_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
submission.head(3)

Sample submission : 3 rows × 235 columns
Our predictions   : 130 rows
Sample row_id sample: BC2026_Test_0001_S05_20250227_010002_5
Our row_id sample  : BC2026_Train_0001_S08_20250606_030007_5
Matched rows      : 0 / 3
Saved: /kaggle/working/submission.csv


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BC2026_Test_0001_S05_20250227_010002_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BC2026_Test_0001_S05_20250227_010002_15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Sanity Checks

In [12]:
# 1. No missing values
assert submission.isnull().sum().sum() == 0, 'Missing values found!'

# 2. All probabilities between 0 and 1
prob_cols = [c for c in submission.columns if c != 'row_id']
assert submission[prob_cols].min().min() >= 0.0, 'Negative probabilities found!'
assert submission[prob_cols].max().max() <= 1.0, 'Probabilities > 1 found!'

# 3. Columns match sample submission exactly
assert list(submission.columns) == expected_columns, 'Column mismatch!'

print('All checks passed ✓')
print(f'  Rows    : {len(submission)}')
print(f'  Columns : {len(submission.columns)}')
print(f'  Prob range: [{submission[prob_cols].min().min():.4f}, {submission[prob_cols].max().max():.4f}]')

All checks passed ✓
  Rows    : 3
  Columns : 235
  Prob range: [0.0000, 0.0000]
